In [1]:
# 面積賃料・平均面積賃料の計算
import pandas as pd

In [2]:
train = pd.read_csv('geocode_train.csv')
test = pd.read_csv('geocode_test.csv')

In [3]:
train1 = pd.read_csv('train.csv')
test1 = pd.read_csv('test.csv')

In [4]:
train1['区'] = train['区']
test1['区'] = test['区']

In [5]:
train1 = train1.drop(columns=['所在地', 'アクセス', '築年数', '方角', '所在階', 'バス・トイレ', 'キッチン', '放送・通信', '室内設備', '契約期間', '駐車場', '周辺環境'])
test1 = test1.drop(columns=['所在地', 'アクセス', '築年数', '方角', '所在階', 'バス・トイレ', 'キッチン', '放送・通信', '室内設備', '契約期間', '駐車場', '周辺環境'])

In [6]:
# trainデータで間取り・区ごとの平均賃料を計算
avg_rent = train1.groupby(['間取り', '区'])['賃料'].mean().reset_index()

In [7]:
# 平均賃料を四捨五入して整数に変換
avg_rent['平均賃料'] = avg_rent['賃料'].round().astype(int)

In [8]:
avg_rent = avg_rent.drop(columns=['賃料'])

In [9]:
# testデータに平均賃料をマージ（左外部結合）
test2 = pd.merge(test1, avg_rent, on=['間取り', '区'], how='left')

In [10]:
# 平均賃料がブランク（NaN）の行に対して同じ区の平均値を埋める
test2['平均賃料'] = test2.groupby('区')['平均賃料'].transform(lambda x: x.fillna(x.mean()))

In [11]:
# 区と間取りごとにグループ化し、平均賃料を計算(train)
train1['平均賃料'] = train1.groupby(['区', '間取り'])['賃料'].transform('mean')

In [12]:
# 平均賃料を四捨五入して整数に変換
train1['平均賃料'] = train1['平均賃料'].round().astype(int)

In [13]:
# 建物構造を数値に変換するマッピング(手動)
location_mapping = {
    'RC（鉄筋コンクリート）': 0,
    'SRC（鉄骨鉄筋コンクリート）': 0,  # コンクリート系にまとめる
    '木造': 1,
    '鉄骨造': 2,
    '軽量鉄骨': 3,  # 鉄骨造にまとめる
    'ALC（軽量気泡コンクリート）': 0,  # コンクリート系にまとめる
    'その他': 4,
    'PC（プレキャスト・コンクリート（鉄筋コンクリート））': 0,  # コンクリート系にまとめる
    'HPC（プレキャスト・コンクリート（重量鉄骨））': 0,  # コンクリート系にまとめる
    'ブロック': 0  # コンクリート系にまとめる
}

In [14]:
# 建物構造を数値に変換
train1['建物構造'] = train1['建物構造'].map(location_mapping)
test2['建物構造'] = test2['建物構造'].map(location_mapping)

In [15]:
# m2を削除して数値に変換
train1['面積'] = train1['面積'].str.replace('m2', '').astype(float)
test2['面積'] = test2['面積'].str.replace('m2', '').astype(float)

In [16]:
train1_drop = train1.drop(columns=['賃料'])

In [17]:
# 2つのデータフレームを結合
df = pd.concat([train1_drop, test2])

In [18]:
# 区と間取りごとにグループ化し、平均面積を計算
df['平均面積'] = df.groupby(['区', '間取り'])['面積'].transform('mean')

In [19]:
# 平均面積を小数点第2位までで揃える
df['平均面積'] = df['平均面積'].round(2)

In [20]:
# 平均賃料/平均面積を計算
df['平均面積賃料'] = df['平均賃料'] / df['平均面積']

In [21]:
# 平均面積賃料を整数で揃える
df['平均面積賃料'] = df['平均面積賃料'].round().astype(int)

In [22]:
# データフレームを分割
train3 = df.iloc[:len(train)].copy()  # trainの行数分だけ抽出
test3 = df.iloc[len(train):].copy()  # testの行数分だけ抽出

In [23]:
train3['賃料'] = train1['賃料']

In [24]:
# 賃料/面積を計算
train3['面積賃料'] = train3['賃料'] / train3['面積']

In [25]:
# 面積賃料を整数で揃える
train3['面積賃料'] = train3['面積賃料'].round().astype(int)

In [26]:
train3 = train3.drop(columns=['区', '間取り', '賃料'])
test3 = test3.drop(columns=['区', '間取り'])

In [27]:
train3['緯度'] = train['緯度']
test3['緯度'] = test['緯度']
train3['経度'] = train['経度']
test3['経度'] = test['経度']

In [28]:
# 緯度・経度を小数点第7位までで揃える
train3['緯度'] = train3['緯度'].round(7)
test3['緯度'] = test3['緯度'].round(7)
train3['経度'] = train3['経度'].round(7)
test3['経度'] = test3['経度'].round(7)

In [29]:
train3.to_csv('train3.csv', index=False)
test3.to_csv('test3.csv', index=False)